# Local LLM Model Evaluation Demo

This notebook asks each local LLM model to write the same Python program many times. Then, it runs each generated program in a clean Python Docker container and counts how often the program works.

This notebook is written for the 2026 UNC Computer Science XL AI Workshop.

The notebook is organized as the experiment story: choose settings, generate programs, run tests, compare results, and inspect failures. The reusable code for generation, Docker execution, evaluation, and display lives in `helper_functions.py` so the notebook can stay focused on the evaluation workflow.

Be sure to run `uv sync` in this project's directory to install required Python library dependencies.

## Settings

This cell is the control panel for the experiment. The list of `MODELS` are those being evaluated. If you were able to run a larger model than `2b`, feel free to add it to the list or replace `2b`.

The model output files are saved to the `generated-programs/` directory, which is ignored by Git.

Each generation uses its attempt counter as the Ollama seed. Seeding controls the randomness of the LLM, so regenerating seed `001` should result in the same output each time, if the prompt, model, and generation settings stay the same.

`SYSTEM_PROMPT` and `PROMPT` are separated so we can discuss the difference between role-level instructions and task-level instructions.

In [31]:
from pathlib import Path

# Experimental Setup:

MODELS = [
    "qwen3.5:0.8b",
    "qwen3.5:9b", # TODO - Enable largest model your machine can run. 
]

SYSTEM_PROMPT = (
    "You are an expert Python programmer. Output only the source code of a complete command-line program, with no explanation and no Markdown. The program reads a single integer from the command-line argument and prints only the result and nothing else."
)

PROMPT = (
    "Write a program that converts an integer between 1 and 1000 inclusive into its Roman numeral. Use standard Roman numeral rules, including the subtractive forms IV for 4, IX for 9, XL for 40, XC for 90, CD for 400, and CM for 900. Read the number from the single command-line argument and print only the uppercase Roman numeral with no other text."
)

N_SAMPLES = 10

TEST_CASES = [
    (["1"], "I"),
    (["3"], "III"),
    (["4"], "IV"),
    (["9"], "IX"),
    (["40"], "XL"),
    (["49"], "XLIX"),
    (["58"], "LVIII"),
    (["90"], "XC"),
    (["99"], "XCIX"),
    (["400"], "CD"),
    (["500"], "D"),
    (["900"], "CM"),
    (["944"], "CMXLIV"),
    (["990"], "CMXC"),
    (["1000"], "M"),
]

# Ollama Generation Parameters
THINK = False
MAX_GENERATED_TOKENS = 1024
TEMPERATURE = 0.2
OUTPUT_DIR = Path("generated-programs")

# Docker Image to Evaluate the Generated Code With
DOCKER_IMAGE = "python:3.12"

In [32]:
print("Configuration loaded")
print("Models:", MODELS)
print("Samples per model:", N_SAMPLES)
print("Output directory:", OUTPUT_DIR)

Configuration loaded
Models: ['qwen3.5:0.8b', 'qwen3.5:9b']
Samples per model: 10
Output directory: generated-programs


## Generate Programs

The next code cell imports the helper functions used by the rest of the notebook. Those helpers are in `helper_functions.py`; the notebook passes in the settings from above so the experiment remains easy to modify here.

The generation cell clears `generated-programs/`, asks each model to generate `N_SAMPLES` programs, and saves each response exactly as the model wrote it. If the model adds Markdown fences or explanation text, the generated `.py` file should fail. That is part of what we are measuring.

In [33]:
from helper_functions import (
    evaluate_model,
    first_failure,
    generate_programs,
    reset_output_dir,
    show_summary_table,
    summarize_results,
)

In [34]:
# This cell will take some time to complete
reset_output_dir(OUTPUT_DIR)

for model_name in MODELS:
    generate_programs(
        model_name=model_name,
        count=N_SAMPLES,
        output_dir=OUTPUT_DIR,
        prompt=PROMPT,
        system_prompt=SYSTEM_PROMPT,
        think=THINK,
        max_generated_tokens=MAX_GENERATED_TOKENS,
        temperature=TEMPERATURE,
    )

Cleared generated-programs/
qwen3.5:0.8b: generating 1
qwen3.5:0.8b attempt 1 - prompt tokens=154, eval tokens=289, duration=5.6s
qwen3.5:0.8b: generating 2
qwen3.5:0.8b attempt 2 - prompt tokens=154, eval tokens=360, duration=2.7s
qwen3.5:0.8b: generating 3
qwen3.5:0.8b attempt 3 - prompt tokens=154, eval tokens=458, duration=3.2s
qwen3.5:0.8b: generating 4
qwen3.5:0.8b attempt 4 - prompt tokens=154, eval tokens=285, duration=2.2s
qwen3.5:0.8b: generating 5
qwen3.5:0.8b attempt 5 - prompt tokens=154, eval tokens=349, duration=2.6s
qwen3.5:0.8b: generating 6
qwen3.5:0.8b attempt 6 - prompt tokens=154, eval tokens=346, duration=2.5s
qwen3.5:0.8b: generating 7
qwen3.5:0.8b attempt 7 - prompt tokens=154, eval tokens=353, duration=2.6s
qwen3.5:0.8b: generating 8
qwen3.5:0.8b attempt 8 - prompt tokens=154, eval tokens=292, duration=2.2s
qwen3.5:0.8b: generating 9
qwen3.5:0.8b attempt 9 - prompt tokens=154, eval tokens=408, duration=2.9s
qwen3.5:0.8b: generating 10
qwen3.5:0.8b attempt 10 - 

In [35]:
print("Current models:", MODELS)

generated_files = sorted(OUTPUT_DIR.rglob("*.py"))
print("Generated file count:", len(generated_files))

for path in generated_files:
    print(path)
    

Current models: ['qwen3.5:0.8b', 'qwen3.5:9b']
Generated file count: 20
generated-programs\qwen3.5-0.8b\001.py
generated-programs\qwen3.5-0.8b\002.py
generated-programs\qwen3.5-0.8b\003.py
generated-programs\qwen3.5-0.8b\004.py
generated-programs\qwen3.5-0.8b\005.py
generated-programs\qwen3.5-0.8b\006.py
generated-programs\qwen3.5-0.8b\007.py
generated-programs\qwen3.5-0.8b\008.py
generated-programs\qwen3.5-0.8b\009.py
generated-programs\qwen3.5-0.8b\010.py
generated-programs\qwen3.5-9b\001.py
generated-programs\qwen3.5-9b\002.py
generated-programs\qwen3.5-9b\003.py
generated-programs\qwen3.5-9b\004.py
generated-programs\qwen3.5-9b\005.py
generated-programs\qwen3.5-9b\006.py
generated-programs\qwen3.5-9b\007.py
generated-programs\qwen3.5-9b\008.py
generated-programs\qwen3.5-9b\009.py
generated-programs\qwen3.5-9b\010.py


In [36]:
import os
import subprocess
import tempfile
import time

import helper_functions


def windows_run_with_bounded_output(
    command,
    timeout_seconds,
    max_output_bytes,
):
    """Windows-compatible subprocess runner with timeout and output limits."""

    with tempfile.TemporaryFile() as stdout_file, tempfile.TemporaryFile() as stderr_file:
        process = subprocess.Popen(
            command,
            stdout=stdout_file,
            stderr=stderr_file,
        )

        started_at = time.monotonic()
        stop_reason = None

        while process.poll() is None:
            elapsed = time.monotonic() - started_at

            if elapsed >= timeout_seconds:
                stop_reason = f"Process timed out after {timeout_seconds} seconds."
                process.kill()
                break

            stdout_size = os.fstat(stdout_file.fileno()).st_size
            stderr_size = os.fstat(stderr_file.fileno()).st_size

            if stdout_size + stderr_size > max_output_bytes:
                stop_reason = (
                    f"Process exceeded the output limit of "
                    f"{max_output_bytes} bytes."
                )
                process.kill()
                break

            time.sleep(0.05)

        process.wait()

        stdout_file.seek(0)
        stderr_file.seek(0)

        stdout_bytes = stdout_file.read(max_output_bytes)
        stderr_bytes = stderr_file.read(max_output_bytes)

    stdout = stdout_bytes.decode("utf-8", errors="replace")
    stderr = stderr_bytes.decode("utf-8", errors="replace")

    # Match Python text-mode subprocess newline handling.
    stdout = stdout.replace("\r\n", "\n").replace("\r", "\n")
    stderr = stderr.replace("\r\n", "\n").replace("\r", "\n")

    if stop_reason is not None:
        if stderr and not stderr.endswith("\n"):
            stderr += "\n"
        stderr += stop_reason

    return subprocess.CompletedProcess(
        args=command,
        returncode=process.returncode,
        stdout=stdout,
        stderr=stderr,
    )


helper_functions._run_with_bounded_output = windows_run_with_bounded_output

print("Installed Windows-compatible evaluation workaround.")


Installed Windows-compatible evaluation workaround.


## Evaluate Programs

The evaluation helpers also live in `helper_functions.py`. Evaluation runs each Python program in an ephemeral Docker container. A generated program succeeds only if it passes every test case with no error and exactly the expected output.

The next code cell evaluates every saved program for every model and stores both the full per-file results and a compact summary.

In [37]:
all_results = {}
summary = {}

for model_name in MODELS:
    rows = evaluate_model(
        model_name=model_name,
        output_dir=OUTPUT_DIR,
        test_cases=TEST_CASES,
        docker_image=DOCKER_IMAGE,
    )
    all_results[model_name] = rows
    summary[model_name] = summarize_results(rows)

summary

{'qwen3.5:0.8b': {'attempts': 10,
  'successes': 0,
  'failures': 10,
  'success_rate': 0.0},
 'qwen3.5:9b': {'attempts': 10,
  'successes': 10,
  'failures': 0,
  'success_rate': 1.0}}

## Compare Models

The summary table shows the main result of the experiment: how often each model produced a working program under the same prompt, test cases, and generation settings.

In [14]:
show_summary_table(summary)

| Model | Attempts | Successes | Failures | Success rate |
| --- | ---: | ---: | ---: | ---: |
| `qwen3.5:0.8b` | 10 | 2 | 8 | 20.0% |
| `qwen3.5:9b` | 10 | 10 | 0 | 100.0% |

## Look At A Few Failures

This cell prints a few failing files and their first failing test. Open the `.py` file to see what the model wrote, then compare the failure with the prompt and system prompt above.

In [38]:
for model_name, rows in all_results.items():
    failures = [row for row in rows if not row["passed"]]
    print(f"\n{model_name}: showing {min(5, len(failures))} of {len(failures)} failures")

    for row in failures[:5]:
        failure = first_failure(row)
        print(row["script"])
        print("  args:", failure["args"])
        print("  expected:", repr(failure["expected"]))
        print("  actual:", repr(failure["actual"]))
        print("  stderr:", failure["stderr"].splitlines()[-1:] or "")


qwen3.5:0.8b: showing 5 of 10 failures
generated-programs\qwen3.5-0.8b\001.py
  args: ['1']
  expected: 'I'
  actual: ''
  stderr: ['Process timed out after 5 seconds.']
generated-programs\qwen3.5-0.8b\002.py
  args: ['1']
  expected: 'I'
  actual: 'Enter an integer between 1 and 1000: '
  stderr: ['EOFError: EOF when reading a line']
generated-programs\qwen3.5-0.8b\003.py
  args: ['1']
  expected: 'I'
  actual: ''
  stderr: ['EOFError: EOF when reading a line']
generated-programs\qwen3.5-0.8b\004.py
  args: ['1']
  expected: 'I'
  actual: ''
  stderr: 
generated-programs\qwen3.5-0.8b\005.py
  args: ['1']
  expected: 'I'
  actual: ''
  stderr: 

qwen3.5:9b: showing 0 of 0 failures
